# 03 · CPI feature engineering

Builds monthly inflation features from the NISR consumer price index (September 2026 release), ready to join with household survey data by month and urban/rural area.

| | |
|---|---|
| **Input** | `data/external/prices/CPI_time_series_September_2026.xls` (read only, never modified) |
| **Output** | `data/processed/cpi_features.parquet` and six charts in `reports/figures/cpi/` |
| **Code** | All logic is in `src/features/cpi_features.py`. This notebook runs it, validates it and explains the results. |
| **Rebuild without the notebook** | `python -m src.features.cpi_features` from the project root |
| **Requirements** | pandas, numpy, xlrd, pyarrow, matplotlib. If reading the `.xls` fails, run `pip install xlrd`. |

**Rules applied**
- **Months only.** The day stored in the workbook header has no meaning, so every observation is dated to the 1st of its month.
- **No future data.** The latest usable month is September 2026. Later months are dropped, and each feature at month *t* uses only months up to *t*. Section 5 tests this.
- **Official values are never changed.** Outliers are flagged in separate columns, not smoothed or removed.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "features" / "cpi_features.py").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import Image, display

from src.features import cpi_features as cf

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)
LATEST = cf.LATEST_MONTH.to_timestamp()  # use timestamps, not strings, when filtering on date
print("Latest usable month:", cf.LATEST_MONTH)

## 1. Load the CPI workbook

The loader reads the **Urban**, **Rural** and **All Rwanda** sheets. It also reads **Other_Indices**, because fresh products, energy, core, local goods and imported goods are only published there. NISR publishes those five indices **for urban areas only**; no rural or national version exists.

While loading, it checks that:
- months are unique and consecutive;
- the 12 COICOP division weights add up to 10,000;
- every series equals 100 in February 2014, the reference period;
- no data after September 2026 is kept.

In [ ]:
long = cf.load_cpi_long()
print(f"{len(long):,} rows | {long['date'].min():%Y-%m} to {long['date'].max():%Y-%m} | {long['date'].nunique()} months")
assert long["date"].max() == cf.LATEST_MONTH.to_timestamp()
long.groupby(["source_sheet", "area"]).agg(series=("category", "nunique"), rows=("index_value", "size"))

In [ ]:
long.head()

## 2. Inflation features

All rates are in percent. Gaps are in percentage points (pp). *I* is the index value.

| Column | Definition |
|---|---|
| `mom_pct` | Month-over-month inflation: 100 × (*I*<sub>t</sub> / *I*<sub>t−1</sub> − 1) |
| `yoy_pct` | Year-over-year inflation: 100 × (*I*<sub>t</sub> / *I*<sub>t−12</sub> − 1) |
| `ann_3m_pct` | 3-month annualised inflation: 100 × ((*I*<sub>t</sub> / *I*<sub>t−3</sub>)<sup>4</sup> − 1). **Not seasonally adjusted.** |
| `cum_change_12m_pct` | 12-month cumulative price change: the last 12 monthly changes compounded. Mathematically identical to `yoy_pct`; the identity is checked below. |
| `avg_annual_pct` | NISR's "annual average inflation": the mean of the last 12 months over the mean of the 12 months before |
| `gap_rural_urban_mom_pp`, `_yoy_pp`, `_ann_3m_pp` | Rural minus urban for the same category and month. The same value appears on every area's row. Empty for urban-only indicators. |
| `mom_robust_z`, `flag_*` | Outlier scores and flags (section 4) |
| `is_indicator` | `True` for the 14 requested indicators (section 3) |

In [ ]:
features = cf.add_inflation_features(long)
assert np.allclose(features["cum_change_12m_pct"], features["yoy_pct"], equal_nan=True, atol=1e-9)

cols = ["area", "index_value", "mom_pct", "yoy_pct", "ann_3m_pct", "cum_change_12m_pct", "avg_annual_pct", "gap_rural_urban_yoy_pp"]
features.query("category == 'headline' and date == @LATEST")[cols]

### Validation against the published report

The table below compares computed figures with the tables in the September 2026 CPI report (`docs/source_documentation/prices/`): 13 months of headline YoY and MoM inflation for all three areas (Table 1), urban annual-average inflation for headline and core (Table 3), and the September figures for the other urban indices (Table 4).

In [ ]:
months = pd.date_range("2025-09-01", "2026-09-01", freq="MS")
published_series = {
    ("urban", "headline", "yoy_pct"): [7.3, 7.1, 7.2, 8.0, 8.9, 9.2, 9.2, 13.0, 12.9, 13.6, 14.5, 15.7, 17.2],
    ("rural", "headline", "yoy_pct"): [5.5, 3.8, 2.0, 3.3, 6.5, 7.0, 6.7, 10.6, 11.8, 12.1, 13.3, 16.0, 22.0],
    ("national", "headline", "yoy_pct"): [6.2, 5.1, 4.1, 5.2, 7.5, 7.9, 7.7, 11.5, 12.3, 12.7, 13.8, 15.9, 20.0],
    ("urban", "headline", "mom_pct"): [1.4, 1.3, 0.2, 0.0, 1.3, 0.9, 1.3, 4.7, 0.5, 0.4, 0.9, 1.8, 2.7],
    ("rural", "headline", "mom_pct"): [1.2, 0.8, -0.3, -0.9, 0.1, 1.4, 1.9, 6.8, 0.7, 0.4, -0.2, 3.1, 6.5],
    ("national", "headline", "mom_pct"): [1.3, 1.0, -0.1, -0.6, 0.6, 1.2, 1.7, 5.9, 0.6, 0.4, 0.3, 2.6, 5.0],
    ("urban", "headline", "avg_annual_pct"): [6.5, 6.7, 6.9, 7.0, 7.2, 7.4, 7.6, 8.2, 8.7, 9.3, 9.9, 10.6, 11.5],
    ("urban", "core", "avg_annual_pct"): [6.3, 6.7, 7.0, 7.2, 7.4, 7.7, 8.0, 8.7, 9.2, 9.7, 10.0, 10.3, 10.5],
}
published_sep_2026 = {  # Table 4: (MoM, YoY)
    "local_goods": (3.4, 19.8), "imported_goods": (0.5, 9.7), "fresh_products": (7.7, 27.9),
    "energy": (0.2, 45.0), "core": (1.3, 11.4),
}
published = [(*key, date, value) for key, values in published_series.items() for date, value in zip(months, values)]
for category, (mom, yoy) in published_sep_2026.items():
    published += [("urban", category, "mom_pct", months[-1], mom), ("urban", category, "yoy_pct", months[-1], yoy)]

check = pd.DataFrame(published, columns=["area", "category", "measure", "date", "published"])
lookup = features.set_index(["area", "category", "date"])
check["computed"] = [lookup.loc[(a, c, d), m] for a, c, m, d in check[["area", "category", "measure", "date"]].itertuples(index=False)]
check["difference_pp"] = (check["computed"].round(1) - check["published"]).abs().round(2)

print(f"{len(check)} published figures compared: {(check['difference_pp'] == 0).sum()} exact, "
      f"largest difference {check['difference_pp'].max():.1f} pp")
assert (check["difference_pp"] <= 0.1).all()
check[check["difference_pp"] > 0]

All but one published figure match exactly. The exception is rural year-over-year inflation for July 2026: 13.24% computed from the workbook against 13.3% in the report. A 0.06 pp gap is too large to be rounding. It most likely reflects a small difference between the index values behind the report and those in the time-series file. It is too small to matter for modelling, but it is noted here.

## 3. Indicators

These 14 indicators are marked `is_indicator = True`. The other categories, such as communication and education, stay in the table because the outlier review needs them.

| Indicator (`category`) | Urban | Rural | National | Source sheet |
|---|:-:|:-:|:-:|---|
| `headline` | ✓ | ✓ | ✓ | Urban / Rural / All Rwanda |
| `food_non_alcoholic_beverages` | ✓ | ✓ | ✓ | " |
| `bread_cereals` | ✓ | ✓ | ✓ | " |
| `meat` | ✓ | ✓ | ✓ | " |
| `milk_cheese_eggs` | ✓ | ✓ | ✓ | " |
| `vegetables` | ✓ | ✓ | ✓ | " |
| `non_alcoholic_beverages` | ✓ | ✓ | ✓ | " |
| `transport` | ✓ | ✓ | ✓ | " |
| `housing_utilities` | ✓ | ✓ | ✓ | " |
| `fresh_products` | ✓ | – | – | Other_Indices |
| `energy` | ✓ | – | – | " |
| `core` (excludes fresh products and energy) | ✓ | – | – | " |
| `local_goods` | ✓ | – | – | " |
| `imported_goods` | ✓ | – | – | " |

September 2026 snapshot:

In [ ]:
latest = features[features["is_indicator"] & features["date"].eq(cf.LATEST_MONTH.to_timestamp())]
snapshot = latest.pivot_table(index="category", columns="area", values=["yoy_pct", "mom_pct", "ann_3m_pct"])
snapshot = snapshot.reindex(cf.INDICATORS)[[(m, a) for m in ["yoy_pct", "mom_pct", "ann_3m_pct"] for a in ["national", "urban", "rural"]]]
snapshot.round(1)

## 4. Outliers and abrupt jumps

Both flags are based on month-over-month inflation:

- **`flag_abrupt_jump`**: absolute MoM change of at least 10%.
- **`flag_outlier`**: robust z-score of at least 6 in absolute value. The score compares this month's MoM change with the median and median absolute deviation (MAD) of the **previous 36 months only**, with at least 24 months required. The scale has a floor of 0.5 pp, so series that rarely move, such as communication and education, do not produce infinite scores.
- **`flag_review`**: either of the two.

These flags mark months to review. They do not mark errors. Each flag uses only past months, so a month's flag never changes when new data arrives.

In [ ]:
n_changes = features["mom_pct"].notna().sum()
flags = features[features["flag_review"]]
print(f"{len(flags)} of {n_changes:,} monthly changes flagged for review ({len(flags) / n_changes:.1%}); "
      f"{features['flag_abrupt_jump'].sum()} abrupt jumps, {features['flag_outlier'].sum()} outliers")

recent = flags[flags["date"] >= "2026-01-01"]
recent[["date", "area", "category", "mom_pct", "mom_robust_z", "flag_abrupt_jump", "flag_outlier"]].sort_values(["date", "area", "category"])

In [ ]:
def is_flagged(area, category, month):
    t = pd.Timestamp(month)
    return bool(features.query("area == @area and category == @category and date == @t")["flag_review"].iloc[0])

assert is_flagged("rural", "communication", "2026-08-01")
assert all(is_flagged(area, "education", "2026-09-01") for area in ["national", "urban", "rural"])

events = features[features["category"].isin(["communication", "education"]) & features["date"].between("2026-06-01", "2026-09-01")]
events.pivot_table(index=["category", "area"], columns=events["date"].dt.strftime("%Y-%m"), values="mom_pct").round(1)

How much did each jump move headline inflation? NISR's modified Laspeyres index is not exactly a weighted sum of its categories; a reconstruction is off by about 1.5 index points out of roughly 250. These contributions are therefore approximate: weight × change in the category index ÷ previous headline index.

In [ ]:
def approx_contribution_pp(area, category, month):
    s = features.query("area == @area and category == @category").set_index("date")
    headline = features.query("area == @area and category == 'headline'").set_index("date")
    t = pd.Timestamp(month)
    prev = t - pd.DateOffset(months=1)
    return s.loc[t, "weight"] / 10_000 * (s.loc[t, "index_value"] - s.loc[prev, "index_value"]) / headline.loc[prev, "index_value"] * 100

rows = []
for area, category, month in [("rural", "communication", "2026-08-01"), ("national", "communication", "2026-08-01"),
                              ("rural", "education", "2026-09-01"), ("national", "education", "2026-09-01"),
                              ("urban", "education", "2026-09-01")]:
    t = pd.Timestamp(month)
    head = features.query("area == @area and category == 'headline' and date == @t")["mom_pct"].iloc[0]
    rows.append({"area": area, "category": category, "month": month[:7],
                 "approx_contribution_pp": approx_contribution_pp(area, category, month), "headline_mom_pct": head})
pd.DataFrame(rows).round(2)

In [ ]:
# Education moves once a year when school fees change: when has it jumped before?
edu = features[features["category"].eq("education") & features["mom_pct"].abs().ge(5)]
edu.pivot_table(index=edu["date"].dt.strftime("%Y-%m"), columns="area", values="mom_pct").round(1)

**Rural communication, August 2026.** The rural communication index rose 54.8% in one month (robust z ≈ 110). It stayed at the new level in September (0.0% change), so this is a step change, not a one-month spike. Urban communication barely moved (−0.3% in August). The pattern looks like a tariff or price change that affected rural areas only; confirm it with NISR before treating it as real. Communication is 1.8% of the rural basket, so the jump added about 0.5 pp to rural monthly inflation in August (3.1% in total). If the new price level holds, rural communication YoY will stay near 55% until July 2027 and fall back in August 2027.

**Education, September 2026.** Education jumped in all three areas: rural +27.6%, national +16.5%, urban +6.8%. The timing is normal. Education prices change once a year with school fees: in January until 2017, in October in 2021 and 2022, and in September since 2024. The size is not normal. This is the largest rural increase since January 2017 (+52.0%). It added about 0.4 pp to rural monthly inflation in September (6.5% in total).

**For modelling:** keep the official values. Use `flag_review` as a control, or leave flagged category-months out when estimating category-specific effects. For a level shift like rural communication, YoY stays high for 12 months, starting with the flagged month.

## 5. No look-ahead check

Rebuild the features using only data up to December 2025. Every value up to that month must be identical to the full build. If any feature used later data, this check would fail.

In [ ]:
truncated = cf.build_cpi_features(latest_month="2025-12")
keys, compare = ["date", "area", "category"], ["index_value", *cf.FEATURE_COLUMNS]
full_part = features[features["date"] <= "2025-12-01"].set_index(keys)[compare].sort_index()
pd.testing.assert_frame_equal(full_part, truncated.set_index(keys)[compare].sort_index())
print(f"{len(full_part):,} rows up to 2025-12 are identical with and without 2026 data")

## 6. Save the features

In [ ]:
path = cf.save_features(features)
reloaded = pd.read_parquet(path)
pd.testing.assert_frame_equal(reloaded, features)
print(f"Saved {len(reloaded):,} rows x {reloaded.shape[1]} columns to {path.relative_to(ROOT)} ({path.stat().st_size / 1e6:.2f} MB)")
reloaded.dtypes.to_frame("dtype").T

## 7. Charts

The charts below are saved to `reports/figures/cpi/`. All show year-over-year inflation from February 2010, the first month with 12 months of history. Line colours are the same in every chart: Rwanda in blue, urban in orange, rural in green.

In [ ]:
chart_paths = {p.stem: p for p in cf.save_charts(features)}
[str(p.relative_to(ROOT)) for p in chart_paths.values()]

In [ ]:
display(Image(filename=str(chart_paths["headline_inflation"])))

**Interpretation.** Inflation accelerated sharply in 2026. National inflation rose from 5.2% in December 2025 to **20.0% in September 2026**, the highest since June 2023. Urban inflation reached 17.2%, the highest since April 2023, and rural inflation 22.0%. The rise came in two steps: an April jump driven by energy and transport prices, then a food-price surge in August and September. Inflation is still below the 2022–23 peak (national 33.8% and rural 42.9% in November 2022). Inflation over the last three months, annualised, is far above the year-over-year rate: 35.6% nationally and 44.1% in rural areas. That suggests inflation had not yet peaked in September.

In [ ]:
display(Image(filename=str(chart_paths["food_inflation"])))

**Interpretation.** Food inflation held at 4–8% through the first half of 2026. It then jumped to about 16–17% in August and, in September, reached **20.9% urban, 25.2% national and 27.0% rural**, the highest since autumn 2023. Food prices usually rise in August and September, but this year the rise was far larger: rural food prices rose 16.1% over the two months, against a 2010–2025 median of 6.0%. Food is 48% of the rural basket against 27% of the urban basket. That is why rural food inflation swings wider in every cycle; at the November 2022 peak it was 73.3% rural against 45.4% urban.

In [ ]:
display(Image(filename=str(chart_paths["fresh_products_inflation"])))

**Interpretation.** Fresh products, the seasonal and most volatile foods, are driving the food surge. Urban fresh-product inflation reached **27.9% in September**, up from 9.1% in June, after monthly rises of 5.8% in August and 7.7% in September. Their combined August–September rise of 14.0% is about twice the usual 7.1% for those months. In past cycles (2012, 2016–17, 2019–20, 2022–23), fresh-product inflation swung two to three times as far as headline inflation in both directions. It is the main source of short-term volatility and the first place to look for a turning point. This index exists for urban areas only.

In [ ]:
display(Image(filename=str(chart_paths["energy_inflation"])))

**Interpretation.** Urban energy inflation is **45.0%**, close to its record of 47.9% in April 2026 and about twice the previous peak of 22.3% in October 2022. Prices rose step by step from late 2025 (year-over-year 14.0% in December), then jumped 18.0% in April 2026. Since May, monthly changes have been close to zero, so the 45% rate reflects that one-off rise in price level, not continuing increases. If energy prices stay flat, energy inflation will drop sharply in April 2027, when the April 2026 increase falls out of the 12-month comparison. Energy is 6.8% of the urban basket, and this index exists for urban areas only.

In [ ]:
display(Image(filename=str(chart_paths["transport_inflation"])))

**Interpretation.** Transport prices move in steps, which is typical of regulated fares and fuel-price adjustments. One large monthly change is followed by flat months, which shows up as a 12-month plateau in year-over-year inflation (see 2020–21 and 2024–25). The latest step came in **April 2026**, when transport prices rose 33.3% in rural areas, 15.1% in urban areas and 19.2% nationally in a single month. That is the largest national monthly increase since May 2020. Year-over-year transport inflation is now 38.7% rural, 22.4% urban and 26.0% national. It should stay near those levels until April 2027 unless fares change again. Transport is 12.5% of the urban basket but only 4.1% of the rural basket, so the larger rural increase has less effect on rural headline inflation.

In [ ]:
display(Image(filename=str(chart_paths["rural_urban_inflation_gap"])))

**Interpretation.** In September 2026, rural inflation was higher than urban inflation by **4.8 pp for headline** and **6.1 pp for food**. These are the widest gaps since September and August 2023 respectively. Between October 2023 and July 2026 the headline gap was negative in all but five months. The gap is mostly driven by food: it closely tracks national food inflation (correlation 0.92 over 2010–2026), because food takes nearly twice the share of the rural basket. Rural households therefore face higher inflation during food-price surges (2012, 2016–17, 2019–20, 2022–23 and now) and lower inflation when food prices fall (2010–11, 2018, 2021 and 2024). For the household analysis, deflate rural and urban spending with their own CPI. Using the national index would understate how much rural households' real incomes fell in periods like this one.

## 8. Wide format for joining with household data

`cf.to_wide()` returns one row per month and area, with one column per indicator and measure (for example `food_non_alcoholic_beverages__yoy_pct`). Join it to the EICV7 data on the interview month and urban/rural status.

In [ ]:
wide = cf.to_wide(features)
print(wide.shape)
wide.query("date == @LATEST").set_index("area")[["headline__yoy_pct", "food_non_alcoholic_beverages__yoy_pct", "transport__yoy_pct", "energy__yoy_pct"]]

## Notes for modelling

- **Urban-only indicators.** Fresh products, energy, core, local goods and imported goods exist only for urban areas, so they are empty on rural and national rows. Using the urban values for rural households is an assumption, and should be stated as one.
- **Weights are from EICV4 (2013/14).** The CPI basket does not reflect EICV7 spending patterns.
- **`ann_3m_pct` is not seasonally adjusted.** Food, fresh products and education are strongly seasonal, so read short-term rates against the same months in earlier years, or use YoY.
- **`cum_change_12m_pct` duplicates `yoy_pct`.** Drop one of them before modelling.
- **Filter the urban rows on `category` or `source_sheet`, not `coicop`.** The `Other_Indices` rows reuse the COICOP codes 01, 04 and 07.